# Custom Benchmark Dataset Generation

---
This notebook implements the custom benchmark dataset generation pipeline

**Importing Required Packages**

In [ ]:
import gzip
import json
from pathlib import Path

from datasets import load_dataset
from tqdm import tqdm

**CONFIG**

In [ ]:
# dataset configuration
DATASET_NAME = "wikimedia/wikipedia"
DATASET_CONFIG = "20231101.en"
SPLIT = "train"

# Output location
OUTPUT_DIR = Path("../data/raw")
OUTPUT_FILE = OUTPUT_DIR / "wikipedia_articles.jsonl.gz"
MAX_ARTICLES = 10000

# Test generator config
DATASET_NAME = "hotpotqa/hotpot_qa"
DATASET_CONFIG = "distractor"
DATASET_SPLIT = "validation"

BENCHMARK_SIZE = 100
RANDOM_SEED = 13

BENCHMARK_DIR = Path("../data/benchmark")
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_FILE = BENCHMARK_DIR / "hotpotqa_subset.jsonl"

**Load hotpotQA Benchmark dataset**

In [ ]:
dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=DATASET_SPLIT
)

print("Total validation examples:", len(dataset))

dataset = dataset.shuffle(seed=RANDOM_SEED)

benchmark_subset = dataset.select(
    range(min(BENCHMARK_SIZE, len(dataset)))
)

print("Selected benchmark examples:", len(benchmark_subset))

In [ ]:
# converting records into benchmark structure
def convert_hotpotqa_record(record):

    context_titles = record["context"]["title"]
    context_sentences = record["context"]["sentences"]

    context_documents = []

    for title, sentences in zip(
        context_titles,
        context_sentences
    ):
        context_documents.append({
            "title": title,
            "text": " ".join(sentences)
        })

    supporting_facts = record["supporting_facts"]

    gold_evidence = []

    for title, sent_id in zip(
        supporting_facts["title"],
        supporting_facts["sent_id"]
    ):

        if title not in context_titles:
            continue

        title_index = context_titles.index(title)
        sentences = context_sentences[title_index]

        if 0 <= sent_id < len(sentences):

            gold_evidence.append({
                "title": title,
                "sentence_id": sent_id,
                "text": sentences[sent_id]
            })

    return {
        "case_id": str(record["id"]),
        "query": record["question"],
        "reference_answer": record["answer"],

        "gold_evidence": gold_evidence,

        "corpus_document_ids": list(
            dict.fromkeys(context_titles)
        ),

        "task_type": record["type"],
        "difficulty": record["level"],

        "context_documents": context_documents
    }

In [ ]:
# create and store benchmark
benchmark_cases = [
    convert_hotpotqa_record(record)
    for record in tqdm(benchmark_subset)
]

with open(BENCHMARK_FILE, "w", encoding="utf-8") as f:

    for case in benchmark_cases:
        f.write(json.dumps(case, ensure_ascii=False) + "\n")

print("Benchmark saved:", BENCHMARK_FILE)
print("Total cases:", len(benchmark_cases))

In [ ]:
# defining a function for retrival and generation

from openai import OpenAI
from dotenv import load_dotenv
import os
import time

load_dotenv()

# get the environment variable from .env file
api_key = os.getenv("OPENROUTER_API_KEY")

# Create a client
llm_client = OpenAI(
    api_key= api_key,
    base_url = "https://openrouter.ai/api/v1"
)

RAG_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"


def run_rag(
    query,
    top_k=5,
    context_override=None
):
    """
    Execute the in-built RAG pipeline.

    Parameters
    ----------
    query : str
        User query.

    top_k : int
        Number of documents to retrieve.

    context_override : list[dict] or None
        If provided, use these documents directly as context
        instead of retrieving documents from ChromaDB.

    Returns
    -------
    dict
        Structured RAG result.
    """

    start_time = time.perf_counter()

    # --------------------------------------------------
    # 1. RETRIEVAL
    # --------------------------------------------------

    if context_override is None:

        query_embedding = sen_trans.encode(
            query,
            normalize_embeddings=True
        )

        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]
        ids = results["ids"][0]

        retrieved_docs = []

        for rank, (doc_id, document, metadata, distance) in enumerate(
            zip(ids, documents, metadatas, distances),
            start=1
        ):

            retrieved_docs.append({
                "id": doc_id,
                "document": document,
                "metadata": metadata,
                "distance": float(distance),
                "rank": rank
            })

        retrieval_performed = True

    else:

        # Use the supplied context instead of retrieving.
        retrieved_docs = context_override

        retrieval_performed = False

    # --------------------------------------------------
    # 2. CONTEXT CONSTRUCTION
    # --------------------------------------------------

    context = "\n\n".join(
        doc["document"]
        for doc in retrieved_docs
    )

    # --------------------------------------------------
    # 3. PROMPT CONSTRUCTION
    # --------------------------------------------------

    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # --------------------------------------------------
    # 4. GENERATION
    # --------------------------------------------------

    response = llm_client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=1024
    )

    answer = response.choices[0].message.content

    # --------------------------------------------------
    # 5. RESULT CONSTRUCTION
    # --------------------------------------------------

    latency = time.perf_counter() - start_time

    effective_context_ids = [
        doc["id"]
        for doc in retrieved_docs
    ]

    usage = getattr(response, "usage", None)

    token_usage = None

    if usage is not None:
        token_usage = {
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "total_tokens": usage.total_tokens
        }

    return {

        "query": query,

        "answer": answer,

        "retrieved_documents": retrieved_docs,

        "latency_seconds": latency,

        "token_usage": token_usage,

        "metadata": {
            "effective_query": query,
            "effective_context_ids": effective_context_ids,
            "applied_retrieval_options": (
                {"top_k": top_k}
                if retrieval_performed else {}
            ),
            "retrieval_performed": retrieval_performed,
            "context_override_used": context_override is not None
        }
    }

In [ ]:
# build an atricle-to-chunk mapping

from collections import defaultdict

BATCH_SIZE = 500

total_chunks = collection.count()

article_chunks = defaultdict(list)

for offset in range(0, total_chunks, BATCH_SIZE):

    batch = collection.get(
        limit=BATCH_SIZE,
        offset=offset,
        include=["metadatas"]
    )

    for chunk_id, metadata in zip(
        batch["ids"],
        batch["metadatas"]
    ):

        if metadata and metadata.get("title"):

            title = metadata["title"]

            article_chunks[title].append(chunk_id)


print("Total chunks:", total_chunks)
print("Unique articles:", len(article_chunks))

In [ ]:
# Sampling different articles for question generation

import random

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

TARGET_ARTICLES = 40

available_titles = list(article_chunks.keys())

selected_titles = random.sample(
    available_titles,
    min(TARGET_ARTICLES, len(available_titles))
)

selected_chunk_ids = []

for title in selected_titles:

    chunk_ids = article_chunks[title]

    selected_chunk_ids.append(
        random.choice(chunk_ids)
    )

print("Selected articles:", len(selected_titles))

In [ ]:
# Retrieving the source passage

selected_records = collection.get(
    ids=selected_chunk_ids,
    include=["documents", "metadatas"]
)

source_passages = []

for chunk_id, document, metadata in zip(
    selected_records["ids"],
    selected_records["documents"],
    selected_records["metadatas"]
):

    if not document:
        continue

    if len(document.strip()) < 300:
        continue

    if len(document) > 3000:
        continue

    source_passages.append({
        "source_id": chunk_id,
        "title": metadata.get("title"),
        "text": document.strip()
    })


print("Eligible source passages:", len(source_passages))

In [ ]:
# function for question generation using sampled passages using LLM

import json
import math

BENCHMARK_GENERATOR_MODEL = RAG_MODEL

def generate_questions_batch(passages):

    input_passages = [
        {
            "source_id": p["source_id"],
            "title": p["title"],
            "passage": p["text"]
        }
        for p in passages
    ]

    prompt = f"""
You are generating benchmark questions for evaluating
a Retrieval-Augmented Generation (RAG) system.

Generate exactly ONE high-quality question for each
provided source passage.

STRICT REQUIREMENTS:

1. The question must be answerable using ONLY its
   corresponding passage.
2. The reference answer must be factually supported
   by that passage.
3. Do not introduce external knowledge.
4. Do not include the answer in the question.
5. Avoid vague questions and questions with ambiguous answers.
6. The evidence_quote must be copied exactly from the passage.
7. Prefer questions that test understanding rather than
   simple copying of a sentence.
8. If a passage does not support a meaningful question,
   mark its status as "skip".

For every passage, return:

- source_id
- status: "ok" or "skip"
- question
- answer
- evidence_quote
- task_type: "factual", "conceptual", or "descriptive"
- difficulty: "easy", "medium", or "hard"

Return ONLY a valid JSON array.

Input passages:

{json.dumps(input_passages, ensure_ascii=False)}
"""

    response = llm_client.chat.completions.create(
        model=BENCHMARK_GENERATOR_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_tokens=2500
    )

    raw_output = response.choices[0].message.content.strip()

    # Remove Markdown code fences if the model adds them
    if raw_output.startswith("```"):
        raw_output = raw_output.split("\n", 1)[1]
        raw_output = raw_output.rsplit("```", 1)[0]

    generated = json.loads(raw_output)

    return generated

In [ ]:
# selecting random 20 passages from the sampled passages

import random

random.seed(42)

generation_sources = random.sample(
    source_passages,
    20
)

print("Selected passages:", len(generation_sources))
print(
    "Unique articles:",
    len({p["title"] for p in generation_sources})
)

In [ ]:
BATCH_SIZE = 4
PILOT_SIZE = 20

# generation_sources = source_passages[:PILOT_SIZE]

generated_candidates = []

for start in range(0, len(generation_sources), BATCH_SIZE):

    batch = generation_sources[
        start:start + BATCH_SIZE
    ]

    print(
        f"Generating questions for passages "
        f"{start + 1} to {start + len(batch)}"
    )

    generated = generate_questions_batch(batch)

    generated_candidates.extend(generated)


print("Generated candidate records:", len(generated_candidates))

In [ ]:
# function for validating generated benchmark questions

import re
import hashlib

def normalize_text(text):

    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)

    return text


source_lookup = {
    p["source_id"]: p
    for p in generation_sources
}

seen_questions = set()


def validate_candidate(candidate):

    errors = []

    source_id = candidate.get("source_id")

    # 1. Source existence
    source = source_lookup.get(source_id)

    if source is None:
        errors.append("unknown_source_id")
        return errors

    question = candidate.get("question", "").strip()
    answer = candidate.get("answer", "").strip()
    evidence_quote = candidate.get("evidence_quote", "").strip()

    # 2. Required fields
    if not question:
        errors.append("missing_question")

    if not answer:
        errors.append("missing_answer")

    if not evidence_quote:
        errors.append("missing_evidence_quote")

    if not question or not answer or not evidence_quote:
        return errors

    # 3. Evidence quote verification
    source_text = source["text"]

    normalized_source = normalize_text(source_text)
    normalized_quote = normalize_text(evidence_quote)

    if normalized_quote not in normalized_source:
        errors.append("evidence_quote_not_found")

    # 4. Answer leakage check
    normalized_question = normalize_text(question)
    normalized_answer = normalize_text(answer)

    if normalized_answer in normalized_question:
        errors.append("possible_answer_leakage")

    # 5. Duplicate question check
    question_fingerprint = normalized_question

    if question_fingerprint in seen_questions:
        errors.append("duplicate_question")

    # 6. Label validation
    valid_task_types = {
        "factual", "conceptual", "descriptive"
    }

    valid_difficulties = {
        "easy", "medium", "hard"
    }

    if candidate.get("task_type") not in valid_task_types:
        errors.append("invalid_task_type")

    if candidate.get("difficulty") not in valid_difficulties:
        errors.append("invalid_difficulty")

    return errors

In [ ]:
validation_results = []

for candidate in generated_candidates:

    if candidate.get("status") != "ok":
        validation_results.append({
            "candidate": candidate,
            "valid": False,
            "errors": ["generation_skipped"]
        })
        continue

    errors = validate_candidate(candidate)

    if not errors:
        seen_questions.add(
            normalize_text(candidate["question"])
        )

    validation_results.append({
        "candidate": candidate,
        "valid": len(errors) == 0,
        "errors": errors
    })


valid_candidates = [
    r["candidate"]
    for r in validation_results
    if r["valid"]
]

invalid_candidates = [
    r for r in validation_results
    if not r["valid"]
]

print("Total candidates:", len(generated_candidates))
print("Valid candidates:", len(valid_candidates))
print("Invalid candidates:", len(invalid_candidates))

In [ ]:
# Converting the candidates into our benchmark schema

import uuid

benchmark_cases_custom = []

# for candidate in approved_candidates:
for candidate in valid_candidates:

    source = source_lookup[candidate["source_id"]]

    case_id = "wiki_" + hashlib.sha1(
        candidate["source_id"].encode()
    ).hexdigest()[:12]

    benchmark_case = {

        "case_id": case_id,

        "query": candidate["question"],

        "reference_answer": candidate["answer"],

        "gold_evidence": [
            {
                "document_id": candidate["source_id"],
                "title": source["title"],
                "text": source["text"],
                "supporting_quote": candidate["evidence_quote"]
            }
        ],

        "gold_evidence_ids": [
            candidate["source_id"]
        ],

        "corpus_document_ids": [
            candidate["source_id"]
        ],

        "task_type": candidate["task_type"],

        "difficulty": candidate["difficulty"],

        "metadata": {
            "source": "Wikipedia",
            "snapshot": "2023-11-01",
            "generation_model": BENCHMARK_GENERATOR_MODEL,
            "generation_method": "LLM-assisted",
            "question_type": "single_hop",
            "human_reviewed": True
        }
    }

    benchmark_cases_custom.append(benchmark_case)


print("Approved benchmark cases:", len(benchmark_cases_custom))

In [ ]:
# saving the benchmarck questions

from pathlib import Path
import json

BENCHMARK_DIR = Path("../data/benchmark")

"""BENCHMARK_DIR = (
    Path.cwd() / "data" / "benchmark"
)"""

BENCHMARK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BENCHMARK_FILE = (
    BENCHMARK_DIR / "wikipedia_custom_benchmark_v1.jsonl"
)

with open(BENCHMARK_FILE, "w", encoding="utf-8") as f:

    for case in benchmark_cases_custom:

        f.write(
            json.dumps(case, ensure_ascii=False) + "\n"
        )


print("Benchmark saved at:", BENCHMARK_FILE.resolve())

In [ ]:
# verifying benchmark against vectorDB

benchmark_evidence_ids = {
    evidence_id
    for case in benchmark_cases_custom
    for evidence_id in case["gold_evidence_ids"]
}

verified_ids = set()

for start in range(
    0, len(benchmark_evidence_ids), 100
):

    batch_ids = list(benchmark_evidence_ids)[
        start:start + 100
    ]

    result = collection.get(
        ids=batch_ids,
        include=[]
    )

    verified_ids.update(result["ids"])


print("Total gold evidence IDs:", len(benchmark_evidence_ids))
print("Verified evidence IDs:", len(verified_ids))
print(
    "Evidence availability:",
    f"{len(verified_ids) / len(benchmark_evidence_ids) * 100:.2f}%"
)

**Establishing Baseline Responses**

In [ ]:
BASELINE_DIR = Path("../data/baselines")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_FILE = BASELINE_DIR / "hotpotqa_baselines.jsonl"

In [ ]:
def run_baseline_experiment(
    benchmark_cases,
    output_file=BASELINE_FILE,
    top_k=5
):

    baseline_results = []

    with open(output_file, "w", encoding="utf-8") as f:

        for case in tqdm(
            benchmark_cases,
            desc="Running baseline RAG"
        ):

            result = run_rag(
                query=case["query"],
                top_k=top_k
            )

            record = {
                "case_id": case["case_id"],
                "baseline": result
            }

            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            baseline_results.append(record)

    print("Baseline experiment completed.")
    print("Total cases:", len(baseline_results))
    print("Saved to:", output_file)

    return baseline_results

In [ ]:
# Load Benchmark dataset

import json
from pathlib import Path

BENCHMARK_FILE = Path("../data/benchmark/wikipedia_custom_benchmark_v1.jsonl")

benchmark_cases = []

with open(BENCHMARK_FILE, "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            benchmark_cases.append(json.loads(line))

print("Total benchmark cases:", len(benchmark_cases))
print("\nSample benchmark case:")
print(json.dumps(benchmark_cases[0], indent=4, ensure_ascii=False))

In [ ]:
# Retrieval function for benchmark questions

TOP_K = 5  

def retrieve_documents(query, top_k=TOP_K):

    # Generate query embedding
    query_embedding = sen_trans.encode(query).tolist()

    # Retrieve relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_documents = []

    for i in range(len(results["ids"][0])):

        retrieved_documents.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved_documents

In [ ]:
# answer generation function

def generate_rag_answer(query, retrieved_documents):

    context = ""

    for i, document in enumerate(retrieved_documents, start=1):

        context += f"""
        Document {i}
        Title: {document["metadata"].get("title", "Unknown")}
        Content:
        {document["text"]}
        """

    system_prompt = """Use the following context to answer the question concisely."""
    #system_prompt = """
    #You are a question-answering assistant.

    #Answer the user's question using only the information
    #available in the provided context.

    #If the context does not contain sufficient information
    #to answer the question, explicitly state that.

    #Do not introduce unsupported information.
    #"""

    user_prompt = f"""
    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm_client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    answer = response.choices[0].message.content

    usage = None
    if response.usage:
        if hasattr(response.usage, "model_dump"):
            usage = response.usage.model_dump()
        else:
            usage = str(response.usage)

    return answer, usage

In [ ]:
# function for calculating retreival metrics

def calculate_retrieval_metrics(retrieved_ids, gold_ids):

    retrieved_ids = retrieved_ids or []
    gold_ids = set(gold_ids or [])

    if not gold_ids:
        return {
            "recall_at_k": None,
            "precision_at_k": None,
            "mrr": None
        }

    retrieved_set = set(retrieved_ids)

    relevant_retrieved = retrieved_set.intersection(gold_ids)

    recall = len(relevant_retrieved) / len(gold_ids)

    precision = (
        len(relevant_retrieved) / len(retrieved_ids)
        if retrieved_ids else 0.0
    )

    reciprocal_rank = 0.0

    for rank, chunk_id in enumerate(retrieved_ids, start=1):

        if chunk_id in gold_ids:
            reciprocal_rank = 1 / rank
            break

    return {
        "recall_at_k": recall,
        "precision_at_k": precision,
        "mrr": reciprocal_rank
    }

In [ ]:
# function for executing the RAG on benchmark questions

import time

def run_baseline(case):

    query = case["query"]
    gold_ids = case.get("gold_evidence_ids", [])

    start_time = time.perf_counter()

    # Step 1: Retrieve documents
    retrieved_documents = retrieve_documents(query)

    # Step 2: Generate answer
    answer, usage = generate_rag_answer(
        query,
        retrieved_documents
    )

    latency = time.perf_counter() - start_time

    # Step 3: Extract retrieved chunk IDs
    retrieved_ids = [
        document["chunk_id"]
        for document in retrieved_documents
    ]

    # Step 4: Calculate retrieval metrics
    metrics = calculate_retrieval_metrics(
        retrieved_ids,
        gold_ids
    )

    # Step 5: Store the complete result
    result = {
        "case_id": case["case_id"],
        "query": query,
        "generated_answer": answer,

        "retrieved_documents": retrieved_documents,
        "retrieved_ids": retrieved_ids,
        "gold_evidence_ids": gold_ids,

        "retrieval_metrics": metrics,

        "execution": {
            "latency_seconds": latency,
            "token_usage": usage,
            "top_k": TOP_K
        }
    }

    return result

In [ ]:
# Executing complete benchmark dataset

baseline_results = []

for i, case in enumerate(benchmark_cases, start=1):

    print(f"\nRunning case {i}/{len(benchmark_cases)}")

    result = run_baseline(case)

    baseline_results.append(result)

    print("Recall@K:", result["retrieval_metrics"]["recall_at_k"])
    print("Precision@K:", result["retrieval_metrics"]["precision_at_k"])
    print("MRR:", result["retrieval_metrics"]["mrr"])

    print("Latency:", round(
        result["execution"]["latency_seconds"], 2
    ), "seconds")

In [ ]:
# save baseline (benchmark) results

RESULTS_DIR = Path("../data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_FILE = RESULTS_DIR / "baseline_results_v1.jsonl"

with open(BASELINE_FILE, "w", encoding="utf-8") as file:

    for result in baseline_results:
        file.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

print("Baseline results saved to:", BASELINE_FILE)

In [ ]:
# calculating aggregate baseline metrics

import numpy as np

recalls = [
    r["retrieval_metrics"]["recall_at_k"]
    for r in baseline_results
    if r["retrieval_metrics"]["recall_at_k"] is not None
]

precisions = [
    r["retrieval_metrics"]["precision_at_k"]
    for r in baseline_results
    if r["retrieval_metrics"]["precision_at_k"] is not None
]

mrr_scores = [
    r["retrieval_metrics"]["mrr"]
    for r in baseline_results
    if r["retrieval_metrics"]["mrr"] is not None
]

latencies = [
    r["execution"]["latency_seconds"]
    for r in baseline_results
]

print("========== BASELINE PERFORMANCE ==========")

print(f"Evaluated cases: {len(baseline_results)}")

print(f"Mean Recall@{TOP_K}: {np.mean(recalls):.4f}")
print(f"Mean Precision@{TOP_K}: {np.mean(precisions):.4f}")
print(f"Mean MRR: {np.mean(mrr_scores):.4f}")

print(f"Mean latency: {np.mean(latencies):.2f} seconds")
print(f"Median latency: {np.median(latencies):.2f} seconds")

--END--